# 자습 11. 로컬 장비에서 SmolVLA 미세조정하기

> **선택 자습 — 본수업 8시간에 포함하지 않습니다.** 코드 셀은 소형 합성 예제이며 GPU 학습·다운로드·로봇 구동을 자동 실행하지 않습니다. 실제 작업 명령은 설명의 터미널 예시로 제공하며 준비물을 먼저 갖춥니다. TODO는 직접 작성하고 아래 정답·해설로 검증하세요. [전체 자습 경로](selfstudy/README.md)


## 자습 중 막혔을 때 — 오류와 조치

먼저 실패한 셀과 전체 오류를 보존하세요. 아래 문구는 대표 예시이며 버전에 따라 달라질 수 있습니다. TODO의 미완료와 실행 환경의 고장을 구분합니다.

| 증상·오류 | 확인 및 조치 | 재확인 기준 |
|---|---|---|
| offline cache 누락·모델 로딩 실패 | 가중치·tokenizer·VLM 설정·processor와 HF_HOME을 확인하고 온라인 준비 후 다시 offline 검사합니다. | 네트워크 없이 로딩 성공 |
| CUDA OOM / `Killed` | batch 1·worker 0과 본인 중복 실행을 확인합니다. RAM 부족과 GPU 부족을 구분합니다. | 작은 역전파·optimizer·저장까지 완료 |
| NaN loss / resume 실패 | 데이터 유한성·정규화와 같은 checkpoint의 train_config/training_state를 확인합니다. | 유한 loss, 저장된 step부터 재개 |
| 20 step 완료인데 작업 실패 | 설치 점검과 학습 완료는 다릅니다. 데이터 품질·학습 규모를 검토하고 12에서 평가합니다. | 동일 조건에서 기준 모델과 비교 |

**예시에 없는 오류를 확인할 곳:** [VLA 단계별 오류 사전](selfstudy/TROUBLESHOOTING.md)의 11 절과 [준비 안내](selfstudy/README.md). 훈련은 터미널 stdout/stderr와 checkpoint 설정, 재생은 지정 output의 결과 파일을 봅니다.

VLA 훈련·저장 관측 재생에는 noVNC 창이 열리지 않습니다. 본수업 GUI rollout을 다시 실행한 경우에만 FINISHED 후 MuJoCo 창을 닫습니다.

[공통 오류 확인 순서와 문의 양식](../자습_오류해결.md)에서 실행 환경·오류 전체·최소 입력·이미 시도한 조치를 정리하세요. 수정 후에는 실패한 검사부터 다시 통과하고 저장 결과까지 확인합니다.


## 실행 전에 준비할 것

**완료 결과:** 재현 가능한 학습 명령, 자원 측정 기록, 저장·재개 검증을 통과한 체크포인트. 자료 학습 90분, 실제 훈련 소요 시간은 장비·데이터에 따라 달라집니다.

10에서 만든 **실제 시연 train dataset** , 로컬로 준비한 사전학습 checkpoint, tokenizer·VLM 설정 캐시, 별도 출력 폴더가 필요합니다. 본수업 배포물에는 자신의 로봇 시연이나 `smolvla_base` 전체 준비물이 자동 포함되어 있지 않습니다. 본수업 합성 예제로 훈련 성공을 주장하지 않습니다.

온라인 준비 단계에서 사용할 모델·데이터 revision과 라이선스를 확인해 다운로드하고 로컬 SHA-256 목록을 기록합니다. 그 후 학습 단계는 아래처럼 offline 설정으로 확인합니다. 설정·tokenizer 캐시가 빠지면 중단하고 준비 단계에서 채웁니다. `load_vlm_weights=False`도 tokenizer·설정 파일 요구까지 없애는 옵션은 아닙니다.

| 환경 | 준비 방법 |
|---|---|
| 기존 Jetson | 별도 자습 이미지·개인 폴더에서 수업의 버전 제약을 유지하고 짧은 backward부터 실측 |
| x86 NVIDIA GPU PC | 해당 CUDA/PyTorch 조합의 별도 환경에 LeRobot 0.4.4 설치·의존성 확인 |
| 버전 변경 | 별도 환경과 데이터 복사본 사용, loader·processor·저장 형식을 재검증 |

설치 명령 하나로 모든 GPU·드라이버 조합을 보장하지 않습니다. 수업 이미지 `nepes-course:physical-ai-vla-v1.0`은 Jetson용이며, 포함된 Jetson 전용 패치를 x86에 그대로 적용하지 않습니다. 학습 중에는 다른 학생의 VLA 추론과 동시에 GPU를 점유하지 않습니다.


## 무엇이 업데이트되나요?

미세조정은 시연 관측으로 예측한 행동을 실제 시연 행동에 맞게 만드는 과정입니다. SmolVLA의 학습 objective는 단순히 로봇 성공/실패를 직접 최적화하는 것과 다릅니다. loss 감소와 rollout 성공을 따로 확인해야 합니다.

첫 시도에서는 설치된 0.4.4가 지원하는 `train_expert_only=True`, `freeze_vision_encoder=True` 설정을 확인합니다. `train_expert_only`는 action expert 중심으로 학습 파라미터를 제한하며 단순 선형 출력층 하나만 학습한다는 뜻은 아닙니다. optimizer에 들어가는 trainable parameter 목록도 확인합니다. frozen 모듈이 있어도 입력 처리·활성값·버퍼·옵티마이저 메모리가 남으므로 작은 메모리를 보장하지 않습니다.

순서는 **데이터 1배치 검사 → 20 step 설치·저장 시범 → 소수 시연 과적합 진단 → 검증 기준을 둔 본 학습** 입니다. 작은 시연조차 loss가 줄지 않으면 장기 실행 전에 단위·통계·카메라·학습률·trainable parameter를 확인합니다. 작은 시연에 과적합되는 것은 학습 경로 확인이지 일반화 성공이 아닙니다.


## 실제 학습 명령 — 준비 완료 후 터미널에서 수동 실행

<!-- procedure:lesson-47-b6cf73:start -->
![긴 학습 전에 작은 경로 검증](../교재그림/절차/lesson-47.svg)

작은 시연도 학습하지 못하면 장기 실행 전에 원인을 찾습니다. loss 감소는 작업 성공을 보장하지 않습니다.
<!-- procedure:lesson-47-b6cf73:end -->


아래 예시는 **LeRobot 0.4.4 기준** 이며 데이터·모델은 사전 준비 항목입니다. `SELFSTUDY_ROOT`는 실제 개인 자습 루트로 바꿉니다. 수업 컨테이너의 Python을 쓰는 경우 `/opt/course/bin/python`을 선택하고, PC에서는 고정된 자습 가상환경의 Python을 사용합니다. `PATH` 설정은 환경에 맞춰 확인합니다. 기존 출력 경로를 삭제하며 재실행하지 말고 새 실행 이름을 사용합니다.

```bash
bash
export SELFSTUDY_ROOT=/workspace/Data/selfstudy/vla
export HF_HOME="$SELFSTUDY_ROOT/cache/huggingface"
export HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1
python -c 'import sys, importlib.metadata as m, torch; print(sys.executable, m.version("lerobot"), torch.cuda.is_available())'
python -m lerobot.scripts.lerobot_train \
  --policy.path="$SELFSTUDY_ROOT/base/smolvla_base" \
  --policy.device=cuda --policy.use_amp=true \
  --policy.load_vlm_weights=false \
  --policy.train_expert_only=true --policy.freeze_vision_encoder=true \
  --policy.input_features='{}' --policy.output_features='{}' \
  --dataset.repo_id=local/pickplace_train \
  --dataset.root="$SELFSTUDY_ROOT/datasets/pickplace_train" \
  --dataset.video_backend=pyav \
  --batch_size=1 --num_workers=0 --steps=20 \
  --eval_freq=0 --save_freq=10 --log_freq=1 \
  --output_dir="$SELFSTUDY_ROOT/train/smoke-001" \
  --job_name=smoke-001 --policy.push_to_hub=false --wandb.enable=false
```

빈 feature 딕셔너리는 데이터 메타데이터에서 키와 shape를 설정하기 위한 것입니다. 단위나 의미를 자동 수정하지 않습니다.

`policy.path`는 시작 가중치, `dataset.root`는 실제 데이터, `output_dir`는 새 저장 경로입니다. `steps=20`은 설치·저장 동작 확인이며 학습 완료 기준이 아닙니다. `eval_freq=0`은 학습 중 자동 환경 평가를 끄고 12에서 별도로 평가하는 설정입니다. 업로드와 W&B 전송은 비활성화합니다.


## 메모리와 속도를 어떻게 조정하나요?

위 명령을 실행하기 전에 데이터 키를 정책이 받을 이름과 대응시키세요. 수업 LIBERO의 camera1/camera2 rename을 자신의 front/wrist 데이터에 무조건 복사하지 않습니다. feature 딕셔너리를 비우는 것은 데이터 메타데이터에서 shape를 정하기 위한 것이며 단위·의미를 보정하지 않습니다.

OOM이면 batch size, 카메라 수·해상도, chunk 길이, trainable module 범위를 순서대로 검토합니다. 설정을 바꾸면 데이터 계약·기존 checkpoint 호환성을 다시 확인합니다. AMP는 실제 accelerator 지원과 수치 유한성을 확인하며 속도·메모리를 측정합니다. 누적 gradient를 쓰고 싶다면 선택한 버전의 공식 지원 경로를 확인하고 effective batch를 기록합니다. 존재를 확인하지 않은 CLI 옵션을 임의로 추가하지 않습니다.

batch 1 성공 후 몇 step을 반복해 최대 메모리·step 시간·온도를 기록합니다. Jetson의 RAM은 CPU와 GPU가 공유하므로 `torch.cuda` 할당량만 보고 전체 메모리 여유를 판단하지 않습니다. 데이터 디코딩이 병목이면 loader worker 수를 늘려 비교하되 프로세스별 메모리도 측정합니다. 카메라 프레임 방향·색순서를 바꿔 해결하지 않습니다.


## TODO — 실제 측정으로 학습 시간 예산 계산

워밍업 이후 측정한 step 시간의 평균과 남은 step 수로 **순수 학습 시간의 근사치** 를 구하세요. 아래 시간은 계산 연습용 합성값입니다. 결과에 환경 초기화·평가·체크포인트 저장·장비 경합은 포함되지 않습니다.


In [ ]:
samples_s = [0.8, 1.0, 1.2]
def estimate_hours(step_seconds, remaining_steps):
    # TODO: 평균 step 시간 × 남은 step 수 / 3600
    return None
hours = estimate_hours(samples_s, 7200)
if hours is None:
    print('TODO 미완료: 시간 추정 함수를 작성하세요.')
else:
    assert abs(hours - 2.0) < 1e-9
    print('합성 예제의 순수 학습 예상 시간:', hours, '시간')

### 정답과 해설

```python
def estimate_hours(step_seconds, remaining_steps):
    if not step_seconds or any(t <= 0 for t in step_seconds) or remaining_steps < 0:
        raise ValueError('유효한 실측 시간과 남은 step 수가 필요합니다.')
    return sum(step_seconds) / len(step_seconds) * remaining_steps / 3600
```

평균은 1초, 7200 step은 7200초이므로 2시간입니다. 이 값을 Jetson 성능 수치로 인용하면 안 됩니다. 자신의 측정값을 넣고, 저장·검증 시간을 별도로 더합니다. batch size를 바꾸면 같은 step 수라도 처리한 표본 수가 달라져 학습량이 달라집니다. 시간만 비교하지 말고 validation 성능과 함께 봅니다.


## 체크포인트와 재개

<!-- procedure:vla-checkpoint-98552c:start -->
![학습 저장과 재개를 검증하기](../교재그림/절차/vla-checkpoint.svg)

추론용 가중치 로드와 optimizer 등을 복구하는 학습 재개는 다릅니다. 먼저 짧은 실행으로 확인합니다.
<!-- procedure:vla-checkpoint-98552c:end -->


저장 디렉터리는 `checkpoints/<step>/pretrained_model/`과 `training_state/`로 나뉩니다. 전자는 정책·설정·processor 파일로 추론에 쓰고, 후자는 optimizer·scheduler·난수·step 복원에 필요합니다. 전체 체크포인트를 보존합니다. 정책 가중치만 다시 읽어 시작하는 것은 정확한 이어 학습과 다릅니다.

아래 명령은 `smoke-001`의 마지막 저장 지점에서 목표 총 step 수를 40으로 늘리는 예시입니다. 실제 저장 폴더에서 `last`가 존재하는지 확인하고 없으면 실제 `<step>` 경로를 씁니다. **재개 명령에는 새 `--policy.path`를 함께 넣지 않습니다.** 0.4.4의 config 파싱 경로가 달라질 수 있기 때문입니다.

```bash
python -m lerobot.scripts.lerobot_train \
  --config_path="$SELFSTUDY_ROOT/train/smoke-001/checkpoints/last/pretrained_model/train_config.json" \
  --resume=true --steps=40
```

**완료 기준:** step이 저장 이후부터 증가, loss·gradient가 유한함, optimizer 상태 복원 로그, 새 checkpoint 저장, 12의 독립 평가 가능. 본 학습에서는 검증 주기와 중단 기준을 먼저 정하고 시험 세트 결과로 매번 설정을 조정하지 않습니다.

문제 대응: 경로 오류는 데이터셋 root와 meta/info.json, offline 캐시 오류는 준비 snapshot, shape 오류는 state/action·processor·padding, NaN은 데이터와 정규화 통계부터 검사합니다. 문서 숫자를 따라 벡터를 임의로 자르지 않습니다.

다음: [12 평가](12_selfstudy_evaluation.ipynb). 명령과 재개 기준은 [0.4.4 학습 소스](https://github.com/huggingface/lerobot/blob/v0.4.4/src/lerobot/scripts/lerobot_train.py), [학습 설정](https://github.com/huggingface/lerobot/blob/v0.4.4/src/lerobot/configs/train.py)을 확인했습니다. 실제 장기 훈련은 이 자료 제작 중 실행하지 않았습니다.
